# Week 32, Day 1: dataclasses

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w032d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from dataclasses import dataclass

@dataclass
class Book:
    title: str
    author: str
    pages: int = 0

a = Book("Aké", "Wole Soyinka", 230)
b = Book("Aké", "Wole Soyinka", 230)
print(a)
print(a == b, a is b, Book("Kindred", "Octavia Butler").pages)

In [ ]:
from dataclasses import dataclass

@dataclass
class Book:
    title: str
    author: str
    pages: int = 0

    def reading_hours(self, pages_per_hour=40):
        return round(self.pages / pages_per_hour, 1)

    def label(self):
        return f"{self.title} by {self.author}"

b = Book("Small Island", "Andrea Levy", 441)
print(b.label(), b.reading_hours(), b.reading_hours(60))
b.pages = 450                  # an ordinary object: attributes can change
print(b)
print(Book(42, None, "lots"))  # annotations aren't checked

In [ ]:
from dataclasses import dataclass

try:
    @dataclass
    class Basket:
        owner: str
        items: list = []
except ValueError as err:
    print("ValueError:", err)

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Basket:
    owner: str
    items: list = field(default_factory=list)
    prices: dict = field(default_factory=dict)

    def add(self, name, pence):
        self.items.append(name)
        self.prices[name] = pence

    def total(self):
        return sum(self.prices.values())

a, b = Basket("Ada"), Basket("Bo")
a.add("yam", 450)
print(a, b, a.total())

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Member:
    name: str
    email: str
    pin: str = field(default="0000", repr=False)
    notes: str = field(default="", compare=False)

m = Member("Tunde", "tunde@example.org", pin="4821", notes="joined at the fair")
print(m)
print(m == Member("Tunde", "tunde@example.org", pin="4821"))

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Reading:
    sensor: str
    celsius: float
    label: str = field(init=False)

    def __post_init__(self):
        self.sensor = self.sensor.strip().lower()
        if not -90 <= self.celsius <= 60:
            raise ValueError(f"impossible temperature: {self.celsius}")
        self.label = f"{self.sensor}: {self.celsius}°C"

r = Reading("  Kitchen ", 21.5)
print(r)
try:
    Reading("roof", 200)
except ValueError as err:
    print("ValueError:", err)

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, order=True, slots=True)
class Version:
    major: int
    minor: int = 0
    patch: int = 0

    @classmethod
    def parse(cls, text):
        return cls(*(int(p) for p in text.split(".")))

    def __str__(self):
        return f"{self.major}.{self.minor}.{self.patch}"

vs = [Version.parse(t) for t in ["1.10", "1.9.2", "2", "1.9"]]
print(sorted(vs))
print(max(vs), Version(1, 9) == Version.parse("1.9.0"), len({Version(2), Version(2, 0, 0)}))
try:
    vs[0].minor = 11
except AttributeError as err:
    print(type(err).__name__, err)

In [ ]:
import json
from dataclasses import dataclass, field, asdict, astuple, replace, fields

@dataclass
class Stop:
    name: str
    minutes: int

@dataclass
class Route:
    code: str
    stops: list = field(default_factory=list)

r = Route("BRT1", [Stop("Ikorodu", 0), Stop("Ketu", 25), Stop("CMS", 70)])
print([f.name for f in fields(r)])
print(asdict(r))
print(json.dumps(asdict(r))[:60] + "...")
print(astuple(Stop("Ketu", 25)), replace(r.stops[1], minutes=30))

> **Key idea.** `@dataclass` reads the fields (`name: type = default`) and writes `__init__`, `__repr__` and `__eq__`; the annotations aren't checked. Fields with defaults come last; use `field(default_factory=list)` for lists and dictionaries, and `field(repr=False, compare=False, init=False)` to fine-tune. `__post_init__` runs after the generated `__init__`, for checks and worked-out fields. `frozen=True` makes objects unchangeable and hashable, `order=True` compares fields in order like tuples, and `slots=True` adds `__slots__`. `asdict` and `replace` convert and copy.

### Exercise: A dataclass book

Turn `Book` into a dataclass with fields `title`, `author` and `year` (default `None`), deleting the hand-written methods that `@dataclass` will now write for you. Keep `age(this_year)`, which gives how many years since the book came out, or `None` if the year isn't known.

In [ ]:
class Book:
    def __init__(self, title, author, year=None):
        self.title = title
        self.author = author
        self.year = year

    def __repr__(self):
        return f"Book(title={self.title!r}, author={self.author!r}, year={self.year!r})"

    def __eq__(self, other):
        if not isinstance(other, Book):
            return NotImplemented
        return (self.title, self.author, self.year) == (other.title, other.author, other.year)

    def age(self, this_year):
        return None if self.year is None else this_year - self.year

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_book():\n    \"a dataclass\"\n    import dataclasses\n    B = ns[\"Book\"]\n    assert dataclasses.is_dataclass(B), \"use @dataclass\"\n    assert \"__repr__\" not in source and \"__eq__\" not in source and \"__init__\" not in source, \"let @dataclass write these\"\n    b = B(\"Things Fall Apart\", \"Chinua Achebe\", 1958)\n    assert repr(b) == \"Book(title='Things Fall Apart', author='Chinua Achebe', year=1958)\"\n    assert b == B(\"Things Fall Apart\", \"Chinua Achebe\", 1958) and b != B(\"Things Fall Apart\", \"Chinua Achebe\")\n    assert b.age(2026) == 68 and B(\"x\", \"y\").age(2026) is None and B(\"x\", \"y\").year is None")

### Exercise: A squad with a list

Write a dataclass `Squad` with `name`, a list `players` (defaulting to a **new** empty list for each squad) and `captain` (default `None`, and left out of the repr). Add `sign(player)`, which adds a player (raising `ValueError` if they're already in the squad), and `size()`.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Squad:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_squad():\n    \"squads\"\n    S = ns[\"Squad\"]\n    a, b = S(\"Reds\"), S(\"Blues\")\n    a.sign(\"Ada\")\n    a.sign(\"Bo\")\n    assert a.players == [\"Ada\", \"Bo\"] and b.players == [] and a.size() == 2 and b.size() == 0\n    assert repr(S(\"Greens\", [\"X\"], captain=\"X\")) == \"Squad(name='Greens', players=['X'])\"\n    assert S(\"Greens\", [\"X\"], captain=\"X\").captain == \"X\" and S(\"G\").captain is None\n    try:\n        a.sign(\"Ada\")\n        assert False, \"signing a player twice should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: Frozen money

Write `Money` as a frozen, ordered dataclass with one field, `pence`. In `__post_init__`, raise `ValueError` unless `pence` is an `int` (not a `bool`). `str` gives `£12.50` (and `-£3.05` for negative amounts). `+` and `-` with another `Money` give a new `Money`; anything else gives `TypeError`. Money must be usable in sets.

In [ ]:
from dataclasses import dataclass

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_money():\n    \"money\"\n    M = ns[\"Money\"]\n    a, b = M(1250), M(305)\n    assert str(a) == \"\u00a312.50\" and str(M(-305)) == \"-\u00a33.05\" and str(M(7)) == \"\u00a30.07\" and repr(a) == \"Money(pence=1250)\"\n    assert a + b == M(1555) and b - a == M(-945) and a > b and sorted([a, b, M(0)]) == [M(0), b, a]\n    assert len({M(5), M(5), M(6)}) == 2\n    try:\n        a.pence = 1\n        assert False, \"Money should be frozen\"\n    except AttributeError:\n        pass\n    for f in [lambda: a + 5, lambda: 5 - a]:\n        try:\n            f()\n            assert False, \"should raise TypeError\"\n        except TypeError:\n            pass\n    for bad in [12.5, \"100\", True]:\n        try:\n            M(bad)\n            assert False, f\"Money({bad!r}) should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Tidying in __post_init__

Write a dataclass `Member` with `name`, `email` and `postcode`, and a field `initials` that isn't an `__init__` argument. In `__post_init__`:

- strip spaces from the ends of all three, and collapse runs of spaces inside the name to one;
- make `email` lower case, and raise `ValueError` unless it has exactly one `@` with something on both sides;
- make `postcode` upper case;
- set `initials` to the first letter of each word of the name, in capitals (`"ada  lovelace"` gives `"AL"`).

Two members are equal when their emails match; `name`, `postcode` and `initials` are left out of the comparison.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Member:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_member():\n    \"tidying\"\n    M = ns[\"Member\"]\n    m = M(\"  ada   lovelace \", \" Ada@Example.ORG \", \" ls1 4ap \")\n    assert (m.name, m.email, m.postcode, m.initials) == (\"ada lovelace\", \"ada@example.org\", \"LS1 4AP\", \"AL\"), f\"got {m}\"\n    assert m == M(\"Ada King\", \"ada@example.org\", \"N1 9GU\") and m != M(\"ada lovelace\", \"ada2@example.org\", \"LS1 4AP\")\n    for bad in [\"no-at-sign\", \"a@@b\", \"@example.org\", \"ada@\", \"a@b@c\"]:\n        try:\n            M(\"x\", bad, \"y\")\n            assert False, f\"email {bad!r} should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        M(\"x\", \"a@b\", \"y\", \"XY\")\n        assert False, \"initials shouldn't be an __init__ argument\"\n    except TypeError:\n        pass")

### Bug hunt: A refused default

The program crashes as soon as the class is defined, with `ValueError: mutable default <class 'list'> for field tracks is not allowed: use default_factory`. Fix it so it prints `[] ['Ye']`.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Playlist:
    name: str
    tracks: list = []

chill, party = Playlist("Chill"), Playlist("Party")
party.tracks.append("Ye")
print(chill.tracks, party.tracks)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"separate lists\"\n    expect_output(\"[] ['Ye']\")")

### Bug hunt: Defaults first

Defining the class crashes with `TypeError: non-default argument 'destination' follows default argument 'origin'`. Every ticket should start in Lagos by default, but the destination must be given. Fix the class (keeping both defaults' meaning) so it prints `Ticket(destination='Abuja', origin='Lagos')`.

In [ ]:
from dataclasses import dataclass

@dataclass
class Ticket:
    origin: str = "Lagos"
    destination: str

print(Ticket(destination="Abuja"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a ticket\"\n    expect_output(\"Ticket(destination='Abuja', origin='Lagos')\")")

### Bug hunt: Changing a frozen score

`Score` is frozen so scores can go in sets, but `add_goal` crashes with `FrozenInstanceError: cannot assign to field 'home'`. Fix `add_goal` so it returns a **new** score instead, and the program prints `Score(home=2, away=1) Score(home=1, away=1)`.

In [ ]:
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Score:
    home: int = 0
    away: int = 0

    def add_goal(self, side):
        if side == "home":
            self.home += 1
        else:
            self.away += 1
        return self

before = Score(1, 1)
after = before.add_goal("home")
print(after, before)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a new score\"\n    expect_output(\"Score(home=2, away=1) Score(home=1, away=1)\")")

### Puzzle: A recipe dataclass

Arrange the lines into a dataclass for a recipe whose ingredients default to a new list each time. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
from dataclasses import dataclass, field

@dataclass(order=True)
class P:
    a: int
    b: int = 0

@dataclass
class Q:
    x: int
    y: int = field(default=5, compare=False)

p = ___   # P(1, 9) < P(2, 0)
q = ___   # P(1, 2) < P(1, 1)
r = ___   # Q(1, 2) == Q(1, 3)
s = ___   # repr(Q(1))
t = ___   # P(3) == P(3, 0)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (True, False, True), \"p to r\"\n    assert s == \"Q(x=1, y=5)\" and t is True")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a diary that saves itself

Write two dataclasses.

**`Event(day, start, title)`**, frozen and ordered.

- `day` is a date as `"YYYY-MM-DD"`; it must be a real date (use `datetime.date.fromisoformat`, which raises `ValueError` for a bad one).
- `start` is a time like `"9:05"` or `"09:05"` (hours 0 to 23, then two digits of minutes 0 to 59), stored as `"09:05"`; otherwise `ValueError`.
- `title` is stored with spaces stripped from the ends, and must not be empty (`ValueError`).
- Because the fields are in the order `day`, `start`, `title`, events sort by day, then time, then title.

**`Diary(owner, events)`**, where `events` defaults to a new empty list.

- `add(event)` adds an event, keeping `events` sorted. An event at the same day and start as one already there raises `ValueError`.
- `on(day)` returns the events on that day, in order.
- `to_json()` returns the diary as JSON text (using `asdict`), and the class method `from_json(text)` makes a diary back from it, so `Diary.from_json(d.to_json()) == d`.

In [ ]:
import json
import re
from datetime import date
from dataclasses import dataclass, field, asdict

___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def E(*a):\n    return ns[\"Event\"](*a)\ndef test_event():\n    \"events\"\n    e = E(\"2026-11-03\", \"9:05\", \"  Dentist \")\n    assert (e.day, e.start, e.title) == (\"2026-11-03\", \"09:05\", \"Dentist\") and repr(e) == \"Event(day='2026-11-03', start='09:05', title='Dentist')\"\n    assert E(\"2026-11-03\", \"10:00\", \"A\") > E(\"2026-11-03\", \"9:30\", \"Z\") and E(\"2026-11-02\", \"23:00\", \"x\") < e\n    assert len({e, E(\"2026-11-03\", \"09:05\", \"Dentist\")}) == 1\n    for bad in [(\"2026-02-30\", \"9:00\", \"x\"), (\"3 Nov\", \"9:00\", \"x\"), (\"2026-11-03\", \"24:00\", \"x\"), (\"2026-11-03\", \"9:5\", \"x\"), (\"2026-11-03\", \"9:00\", \"   \")]:\n        try:\n            E(*bad)\n            assert False, f\"Event{bad} should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        e.title = \"x\"\n        assert False, \"events are frozen\"\n    except AttributeError:\n        pass\ndef test_diary():\n    \"the diary\"\n    D = ns[\"Diary\"]\n    d, other = D(\"Ngozi\"), D(\"Femi\")\n    d.add(E(\"2026-11-04\", \"14:00\", \"Football\"))\n    d.add(E(\"2026-11-03\", \"9:00\", \"Dentist\"))\n    d.add(E(\"2026-11-03\", \"8:30\", \"Run\"))\n    assert [ev.title for ev in d.events] == [\"Run\", \"Dentist\", \"Football\"] and other.events == []\n    assert [ev.title for ev in d.on(\"2026-11-03\")] == [\"Run\", \"Dentist\"] and d.on(\"2026-12-25\") == []\n    try:\n        d.add(E(\"2026-11-03\", \"09:00\", \"Haircut\"))\n        assert False, \"a clash should raise ValueError\"\n    except ValueError:\n        pass\n    assert len(d.events) == 3\ndef test_json():\n    \"saving and loading\"\n    import json as js\n    D = ns[\"Diary\"]\n    d = D(\"Ngozi\")\n    d.add(E(\"2026-11-03\", \"9:00\", \"Dentist\"))\n    text = d.to_json()\n    data = js.loads(text)\n    assert data == {\"owner\": \"Ngozi\", \"events\": [{\"day\": \"2026-11-03\", \"start\": \"09:00\", \"title\": \"Dentist\"}]}, f\"got {data}\"\n    back = D.from_json(text)\n    assert back == d and isinstance(back.events[0], ns[\"Event\"]) and D.from_json(D(\"x\").to_json()) == D(\"x\")")

## Warm-up

The class body has no methods at all, just three lines naming the fields, yet `Book` has an `__init__` taking three arguments (the last one optional), a readable `__repr__`, and an `__eq__` that compares by value. `@dataclass` wrote them. That's today's tool: you describe the data, and Python writes the routine methods.

## What @dataclass writes

`@dataclass` is a **class decorator**, like `@total_ordering` last week: written above a class, it changes the class after Python has made it. (Phase 4 explains how decorators work; for now, they're switches you put above things.) It reads the class's **fields**, the lines of the form `name: type`, and writes:

| Method | What it does |
|---|---|
| `__init__` | takes the fields in order, as arguments, and stores them |
| `__repr__` | `Book(title='Aké', author='Wole Soyinka', pages=230)` |
| `__eq__` | equal when the other object has the same class and all fields are equal |

The `: str` and `: int` parts are **type annotations**. They're required here, because the colon is how `@dataclass` tells a field from an ordinary class attribute. But Python doesn't check them: `Book(42, None, "lots")` works without complaint. Week 33 is about type hints and the tools that do check them; today, read `pages: int` as "pages, which should be an int".

You can still write any methods you like, and they work as in any class:

If you write one of the generated methods yourself (say, your own `__repr__`), `@dataclass` leaves yours alone. And a dataclass is a normal class in every other way: it can inherit, be inherited from, have properties and class methods.

Notice one thing that's missing: with `__eq__` written and no `__hash__`, an ordinary dataclass is **unhashable** (Week 29's rule: equality based on changeable values must not be hashed). Frozen dataclasses, below, fix that.

## Defaults and field()

Fields with defaults must come **after** fields without, because they become `__init__`'s parameters in order, and Python doesn't allow a required parameter after an optional one. And there's one default `@dataclass` refuses outright:

That's Week 28's shared-mutable-default trap, and `@dataclass` catches it for lists, dictionaries and sets. The fix is `field(default_factory=list)`: instead of one shared list, `default_factory` is called (here, `list()`) to make a fresh one for each object.

`field(...)` takes other options too. The most useful:

| Option | Meaning |
|---|---|
| `default=...` | an ordinary default value |
| `default_factory=f` | call `f()` for each new object |
| `repr=False` | leave this field out of the repr (for long or private data) |
| `compare=False` | leave it out of `==` (and ordering and hashing) |
| `init=False` | not an `__init__` argument; set it yourself, usually in `__post_init__` |

## Checking values in __post_init__

The generated `__init__` stores whatever it's given. To check or tidy values, write `__post_init__`: the generated `__init__` calls it as its last step, after every field is set.

`label` has `init=False`, so it isn't an argument; `__post_init__` works it out from the other fields. Keep the checks here, in one place, as you did in hand-written `__init__` methods.

Remember that `__post_init__` runs only when the object is **made**. Changing `r.celsius = 500` afterwards isn't checked. If that matters, make the class frozen, or use a property (Week 29) for that attribute.

## Frozen, ordered and slotted

`@dataclass` takes options that switch on more generated code:

| Option | What you get |
|---|---|
| `frozen=True` | assigning to any field raises `FrozenInstanceError` (a kind of `AttributeError`), and `__hash__` is written, so objects can go in sets and be dictionary keys |
| `order=True` | `<`, `<=`, `>`, `>=`, comparing the fields **in order**, like tuples |
| `slots=True` | a `__slots__` with the field names (Week 31, Day 5) |

Together, they make a value class in a few lines. Compare this with Week 29's hand-written `Money`, or Week 31's `Version`:

- `order=True` compares `(major, minor, patch)` as tuples, so the **order of the fields** is the order of importance. Choose it deliberately.
- Frozen objects can't be changed, even in `__post_init__`. To tidy a value there, use `object.__setattr__(self, "name", value)`, exactly as in Week 31.
- A frozen object is "changed" by making a new one: `dataclasses.replace(v, minor=11)` copies `v` with one field different.

## Tools for dataclasses

The `dataclasses` module has a few helpers that work on any dataclass object:

`asdict` turns a dataclass into a dictionary, going **inside** lists and other dataclasses too, which makes saving to JSON (Week 23) a one-liner. `replace` makes a copy with some fields changed (and runs `__post_init__` again, so its checks still apply).

When to use a dataclass: almost any class that is mainly **data**, with some methods. When a class has complicated rules about its attributes, or its `__init__` takes arguments that aren't simply stored, a hand-written class can still be clearer. And Day 3 compares dataclasses with named tuples.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick a class you wrote by hand in Weeks 28 to 31 and rewrite it as a dataclass. Count the lines you saved, and check whether any behaviour changed (look especially at hashing and ordering). Tomorrow: Enums, for values that come from a fixed set of choices.